# Run and plot a model in Python

[Choose a model](../MODEL_CATALOG.md) · [Understand results](understand-results.md) · [Change an input](change-inputs.md)

Complete the [README setup](../README.md#setup), activate that environment,
and run the blocks below from the library checkout. Core simulation needs no
plotting packages.

This notebook locates the library checkout automatically. CSV output is saved
in the notebook’s working directory.

## Run the first experiment

In [ ]:
from qspquest_models import Library

from pathlib import Path
root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'catalog.json').is_file())
library = Library(root)
print(len(library.list_models()))  # 72 identities
model = library.load('qspquest_serkland_2025_rituximab_ms_pkpd', version='0.3.0')
print(model.scenario_ids)
result = model.run('population-typical-1000mg-full-ode')
assert len(result['times']) == 201
model.write_csv(result, 'rituximab-notebook.csv')
print('Saved rituximab-notebook.csv with', len(result['times']), 'rows')

`result` contains `times`, event `sides`, named `outputs` and `states`, units,
archive checksum and engine identity. The CSV has 201 data rows from day 0 to day 200 with `time`, `side`,
`rituximab_concentration_mg_l`, `cd19_cells_mm3` as the primary readouts. The [model guide](../models/qspquest_serkland_2025_rituximab_ms_pkpd/0.3.0/MODEL_CARD.md) explains the available experiments.
[Output interpretation](understand-results.md) explains the source numerical scale and units.

## Plot CD19+ B cells (optional)

Install matplotlib into the same environment with `python -m pip install matplotlib`.
Then continue in the same Python session:

In [ ]:
import matplotlib.pyplot as plt

cd19_cells_mm3 = next(c for c in result['outputs'] if c['name'] == 'cd19_cells_mm3')
plt.plot(result['times'], cd19_cells_mm3['values'])
plt.xlabel('Time (' + result['time_unit'] + ')')
plt.ylabel('CD19+ B cells (source scale)')
plt.show()

![Stored rituximab B-cell trajectory](rituximab-bcells.svg)

This preview uses the retained version 0.1.0 result. Its numerical reference
applies to that archive and recorded execution; inspect the selected version’s
`evidence.json` before treating a new run as qualified.

The B-cell readout starts at 199.57 and ends near 8.70521 at day 200.
This curve describes a fixed typical parameter set, rather than population variability
or a recommended redosing time.

## Try a synthetic benchmark

In [ ]:
model = library.load('qspquest_diffdose_2026_idr_benchmark')
result = model.run('lower_exposure')

## Find your next model

Browse the [model catalog](../MODEL_CATALOG.md) for scientific descriptions and
experiment choices. In Python, you can filter the structured records:

In [ ]:
matches = [row for row in library.list_models()
           if 'rituximab' in row['name'].lower() and row['scenario_ids']]
for row in matches:
    print(row['name'], row['id'])

Continue with [comparing experiments or changing a dose](change-inputs.md).
For exact version selection and archive inspection, use the [technical reference](model-format.md).

The selected model requires the matching QSPQuest count-unit engine. Returned rituximab concentration uses `mg/L`; CD19-positive cell density uses `[cell]/mm3`. Dimensional, numerical and biological qualification remain separate and pending.
